# Week 3 — The Retrieval LabWeek 2 ended with one honest gap. Your context template had a `sources=` block, andyou filled it by hand — which only works if you already know which passage answers thequestion. This session builds the thing that fills it, and then closes the loop byhanding the result back to the model.**How to work through this.** Run the cells in order, on **your own corpus**. Everynumber you get should come from material you chose, not from the example below.**On cost, and this one bites.** Embeddings are rate-limited to roughly **100 texts perminute**, counted per text rather than per request — batching cannot speed it up, onlywaiting can. The sample corpus below is small enough to embed in the notebook. **Yourreal corpus is not:** embed it once with `python ingest.py`, which paces itself andwrites to `./chroma/`, and it stays embedded across every kernel restart afterwards.Generation, in Parts 6 and 7, is the separate **6 calls per minute** limit.**Before you start:** you need your Week 2 context template — `SYSTEM`, `TASK`,`FORMAT`. If you did not keep it, reconstruct it now; Part 6 depends on it.

In [ ]:
from chunking import chunk_textfrom embedding_client import EmbeddingClientfrom vector_store import VectorStoreemb = EmbeddingClient()print(f"embeddings: {emb.provider} / {emb.model}")

---## Part 1 — Your corpus, in piecesReplace `MY_CORPUS` with real text from the domain you proposed in Week 1. Paste two orthree documents' worth — a few hundred words is enough to make the chunking visible,and you will point this at the real thing for Lab 1.

In [ ]:
MY_CORPUS = """Модель може працювати лише з тим, що є в її контексті. Усе, що потрапляє у вікноконтексту, оплачується під час кожного виклику. Пошук потрібен для того, щобпрограма знаходила потрібний фрагмент, а розробник не мусив заздалегідь знати,який саме абзац відповідає на запитання.Lab 2 - Knowledge graph and Graph RAG - due end of Week 7 - 25% of the final grade.Deliverables: Neo4j database and graph model; re-seedable ingest.py; Cypher queries;graph retrieval implementation; measured comparison against the Lab 1 system on theshared gold set; README with AI-use disclosure.Chroma's default distance is not cosine. An unconfigured collection uses squaredEuclidean distance, so a collection must be created with hnsw:space set to cosineto match the metric this course teaches."""chunks = chunk_text(MY_CORPUS, chunk_size=60, overlap=15)print(f"{len(MY_CORPUS.split())} words -> {len(chunks)} chunks\n")for i, c in enumerate(chunks):    print(f"[{i}] {c[:90]}...")

Now change `chunk_size` and `overlap` and run it again. Watch what happens to asentence that sits on a boundary.**Write down the size you settle on and why.** There is no correct answer — too smallloses the context that gave a chunk its meaning, too large dilutes relevance withunrelated material. You will defend this choice in Lab 1.

### ✍️ Your notes**Chunk size and overlap I chose:****Why:**

---## Part 2 — Look at an actual embeddingThe lecture claimed two things about the vector. Check both yourself rather thantaking them on faith.

In [ ]:
short = emb.embed("контекст")long_ = emb.embed(" ".join(chunks))print(f"1 word   -> {len(short)} numbers")print(f"{len(' '.join(chunks).split()):>3} words -> {len(long_)} numbers")print(f"\nsame length? {len(short) == len(long_)}")print(f"\nfirst 8 numbers: {[round(x, 4) for x in short[:8]]}")

**Claim 1: fixed length regardless of input.** Confirmed above.**Claim 2: nearby means related.** The numbers themselves are unreadable — what makesthem useful is how they compare. Pick a question about your corpus, and a sentence thathas nothing to do with it.

In [ ]:
QUESTION = "Що оплачується під час кожного виклику?"UNRELATED = "Рецепт хліба починається з борошна та води."q = emb.embed(QUESTION, task_type="RETRIEVAL_QUERY")relevant = emb.embed(chunks[0], task_type="RETRIEVAL_DOCUMENT")unrelated = emb.embed(UNRELATED, task_type="RETRIEVAL_DOCUMENT")print(f"cosine(relevant chunk, question)  = {EmbeddingClient.cosine(relevant, q):.4f}")print(f"cosine(unrelated text, question)  = {EmbeddingClient.cosine(unrelated, q):.4f}")

The unrelated score is probably **not** near zero — two arbitrary sentences in the samelanguage share a lot of structure. That is fine. Retrieval never needs an absolutethreshold; it needs the relevant one to rank **higher**, which is what you just checked.

---## Part 3 — Retrieval, end to end`VectorStore` is the ingest-once, query-every-call split from the lecture. `add()` isingestion. `search()` runs per question.Two things about it are deliberate. The collection is created with`hnsw:space = "cosine"` — Chroma's default is squared Euclidean, which is **not** whatthis course measures. And the store is **persistent**: it writes to `./chroma/` andsurvives a kernel restart, which is what makes a rate-limited ingest survivable.The sample below is a handful of chunks, so embedding it here is fine.

In [ ]:
store = VectorStore(embedder=emb, name="lab_demo", reset=True)store.add(chunks)print(f"{store.count()} chunks indexed\n")for row in store.search(QUESTION, n_results=3):    print(f"distance={row['distance']:.4f}  {row['chunk'][:80]}...")

`search()` shows what is *relevant*. To see what is actually stored — which is how youcheck a chunk boundary — look directly:

In [ ]:
print(f"{store.count()} chunks in {store.collections()}\n")for row in store.peek(3):    print(f"[{row['id']}] {row['chunk'][:100]}...")

Read chunk `[0]` against your source text. Did the boundary land mid-sentence? That is thetrade-off from Part 1, now visible in what you actually stored rather than argued about ona slide.From a terminal you can browse the whole collection instead:`chroma browse chunks --path ./chroma` — arrows to move, `Return` to expand, `q` to quit.

**For your own corpus, stop and do this in a terminal instead:**```bashpython ingest.py corpus/ --chunk-size 200 --overlap 40```Then come back and open what it built — no re-embedding, however often you restart:```pythonstore = VectorStore(name="chunks")     # already populatedprint(store.count())```Re-run `ingest.py --reset` only when you change the chunk size, since that invalidatesevery boundary already stored.

`distance` here is **1 − cosine similarity**, so smaller is closer.Try three or four more questions about your own corpus. Look for one it gets right andone it gets wrong — you need both for Part 6.

### ✍️ Your notes**A question it answered well:****A question it got wrong, and what it returned instead:**

---## Part 4 — The gold set (due this week)Ten representative queries for your domain, each with something checkable: adistinctive phrase that the *correct* chunk must contain.**Write these before you tune anything.** A gold set written afterwards, to fit resultsyou already have, measures nothing. This is the seed every later improvement in thecourse is compared against — including Week 4, next session.

In [ ]:
GOLD = [    {"query": "Що оплачується під час кожного виклику?", "expect": "оплачується"},    {"query": "What is Lab 2 worth?", "expect": "25%"},    {"query": "Which distance does Chroma use by default?", "expect": "Euclidean"},    # ... write seven more, for YOUR corpus.]print(f"{len(GOLD)} queries written. Target: 10.")

In [ ]:
def hits_at_k(store, gold, k=5):    """How many gold queries return the right chunk in their top k.    A rough count, not yet a measurement -- Week 4 turns this into recall@k    properly, across several values of k and with the failures diagnosed.    """    found = 0    for item in gold:        results = store.search(item["query"], n_results=k)        if any(item["expect"].lower() in r["chunk"].lower() for r in results):            found += 1    return foundprint(f"{emb.model}: {hits_at_k(store, GOLD)} / {len(GOLD)} found in top 5")

---## Part 5 — Choosing a model, measured *(optional)*The lecture argued model choice in the abstract. Here it is on your corpus, settled bya number instead of a preference.`gemini-embedding-001` against OpenRouter's `baai/bge-m3` — both multilingual, bothplausible for a Ukrainian corpus. **Needs `OPENROUTER_API_KEY` in `.env`.**Skip this part if the session is short; it is the one section designed to be dropped.On a real corpus, run it over a **subset** — every comparison costs a second fullingest against a 100-texts-per-minute limit.

In [ ]:
alt = EmbeddingClient(provider="openrouter")alt_store = VectorStore(embedder=alt, name="lab_demo_alt", reset=True)alt_store.add(chunks)print(f"{emb.model:<24} {hits_at_k(store, GOLD)} / {len(GOLD)}")print(f"{alt.model:<24} {hits_at_k(alt_store, GOLD)} / {len(GOLD)}")

**Discuss whichever way it lands.** A tie is the most likely result on a small corpus,and a tie is itself an answer: it means the choice should be made on the things that arenot retrieval quality — rate limit, cost, dimensionality and therefore storage, andwhether `task_type` works at all.That last one is not a tiebreaker, it is decisive. Query/document asymmetry works on theGemini backend and is a confirmed **no-op** through OpenRouter, even routing the verysame model. Check `embedding_client.py`'s capability matrix before you choose.

### ✍️ Your notes**Which model I will use for Lab 1, and the number that decided it:**

---## Part 6 — Hand the chunks to the modelThis is the loop closing. Everything below is your Week 2 template — the only new lineis the one that puts retrieved text where pasted text used to go.

In [ ]:
from context_lab import askSYSTEM = (    "You answer questions using only the sources you are given. If the sources do "    "not answer the question, reply exactly: not stated. Never fill a gap from your "    "own knowledge.")TASK = "Answer the question using only the SOURCES above."FORMAT = "Two sentences maximum. Put the source number in square brackets after each claim."def rag(question, *, k=3, show_prompt=True):    """Retrieve, assemble, generate. The whole of RAG, in four lines."""    retrieved = [row["chunk"] for row in store.search(question, n_results=k)]    return ask(question, system=SYSTEM, task=TASK, fmt=FORMAT,               sources=retrieved, show_prompt=show_prompt, label=f"rag k={k}")result = rag(QUESTION)

Read the printed context before you read the answer. The `SOURCES` block was assembledby your code, from your corpus, moments ago — nothing else in the template changedsince Week 2.Now run the question Part 3 got **wrong**. Retrieval failing means the model isgrounded in the wrong passage, and a wrong passage does not announce itself.

In [ ]:
# The question your retrieval handled badly. Read the cited chunk, not just the answer.BAD = "..."   # replace with your own# result = rag(BAD)

**Find at least one case where the answer reads well and the cited chunk does notsupport it.** That case is the entire reason Week 4 exists, and you will need it nextsession.

### ✍️ Your notes**A fluent answer whose source did not support it:****Where it went wrong — chunking, retrieval, or generation?**

---## Part 7 — The decisions that are yoursThe lecture listed four: how many chunks, in what order, truncated how, numbered how.Here is what the first one costs.

In [ ]:
from context_lab import counttokens, renderfor k in (1, 3, 5, 10):    retrieved = [row["chunk"] for row in store.search(QUESTION, n_results=k)]    prompt = render(QUESTION, task=TASK, fmt=FORMAT, sources=retrieved)    print(f"  k={k:>2}  ->  {counttokens(prompt):>5} prompt tokens, every call")

Every one of those tokens is billed on **every** call — the Week 2 arithmetic, arrivingsomewhere you did not expect it. Retrieving ten chunks where three would do is a costdecision you make by accident unless you look.Run `rag(QUESTION, k=1)` and `rag(QUESTION, k=10)` and compare the answers, not just theprices. More context is not reliably better: the relevant passage now competes withmaterial that is merely nearby.**The other three decisions** are yours to explore in Lab 1. Ordering: `search()` returnsnearest first — is that the best position for the strongest chunk? Truncation: what doyou cut when the budget is exceeded? Numbering: `render()` writes `[1]`, `[2]`, which isthe only reason a citation can be checked at all.

### ✍️ Your notes**The k I chose, and what it cost:**

---## Before you close this**Due this week:** your ten-query gold set, and a program that takes a question andreturns a grounded answer from your own corpus.**What you still do not know:** how often it is right. You have checked a handful ofqueries by eye and measured one choice against the gold set. That is not the same asknowing the system works, and the fluent-but-unsupported answer you found in Part 6 isthe proof that reading outputs will not settle it.**Week 4 measures it** — recall@k across the whole gold set, the failure modes behindeach miss, and how to tell which stage went wrong.

In [ ]:
from context_lab import callsprint(f"generation calls this session: {calls()}")print(f"embedding requests:            {emb.calls}")print(f"texts embedded:                {emb.texts_embedded}  (limit ~100/minute)")